# 04 — Evaluation: Model Rekomendasi Latihan

**Comprehensive evaluation:**
1. Confusion matrix per output (workout_type + intensity)
2. Classification report dengan precision/recall/F1 per kelas
3. ROC curves (one-vs-rest) untuk multi-class
4. RF vs XGBoost head-to-head comparison
5. Top-15 feature importance interpretation

**Target:** F1-macro workout ≥ 0.85, intensity ≥ 0.80

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

os.makedirs('output/evaluation', exist_ok=True)

# Load models
model_workout = joblib.load('output/models/workout_xgb_workout_type.pkl')
model_intensity = joblib.load('output/models/workout_xgb_intensity.pkl')

# Load data
X_test = pd.read_parquet('output/preprocessed/X_test_scaled.parquet')
yw_test = pd.read_parquet('output/preprocessed/yw_test.parquet')['workout_type_label']
yi_test = pd.read_parquet('output/preprocessed/yi_test.parquet')['intensity_label']

with open('output/preprocessed/metadata.json') as f:
    meta = json.load(f)

WORKOUT_LABELS = meta['workout_type_map']  # {'0': 'FLEXIBILITY', ...}
INTENSITY_LABELS = meta['intensity_map']

print(f'Test set: {X_test.shape}')
print(f'Workout classes: {sorted(yw_test.unique())}')
print(f'Intensity classes: {sorted(yi_test.unique())}')

In [ ]:
# ── Predict ──
yw_pred = model_workout.predict(X_test)
yi_pred = model_intensity.predict(X_test) + 1  # back from 0,1,2 → 1,2,3

# Probabilities for ROC
yw_proba = model_workout.predict_proba(X_test)
yi_proba = model_intensity.predict_proba(X_test)

print('Predictions completed.')

In [ ]:
# ── 1. Overall Metrics ──
acc_w = accuracy_score(yw_test, yw_pred)
acc_i = accuracy_score(yi_test, yi_pred)
f1_w = f1_score(yw_test, yw_pred, average='macro', zero_division=0)
f1_i = f1_score(yi_test, yi_pred, average='macro', zero_division=0)
f1_w_weighted = f1_score(yw_test, yw_pred, average='weighted', zero_division=0)
f1_i_weighted = f1_score(yi_test, yi_pred, average='weighted', zero_division=0)

print('='*60)
print('OVERALL METRICS — Model Rekomendasi Latihan (XGBoost)')
print('='*60)
print(f'  Workout Type:')
print(f'    Accuracy:      {acc_w:.4f}')
print(f'    F1-macro:      {f1_w:.4f}  (target ≥ 0.85)')
print(f'    F1-weighted:   {f1_w_weighted:.4f}')
print(f'  Intensity Band:')
print(f'    Accuracy:      {acc_i:.4f}')
print(f'    F1-macro:      {f1_i:.4f}  (target ≥ 0.80)')
print(f'    F1-weighted:   {f1_i_weighted:.4f}')
print()
print(f'  {"✅" if f1_w >= 0.85 else "❌"} Workout F1 target')
print(f'  {"✅" if f1_i >= 0.80 else "❌"} Intensity F1 target')

In [ ]:
# ── 2. Classification Report ──
workout_classes = sorted(yw_test.unique())
intensity_classes = sorted(yi_test.unique())
workout_names = [WORKOUT_LABELS[str(k)] for k in workout_classes]
intensity_names = [INTENSITY_LABELS[str(k)] for k in intensity_classes]

report_w = classification_report(yw_test, yw_pred, target_names=workout_names,
                                  zero_division=0, digits=4)
report_i = classification_report(yi_test, yi_pred, target_names=intensity_names,
                                  zero_division=0, digits=4)

print('--- WORKOUT TYPE ---')
print(report_w)
print('--- INTENSITY BAND ---')
print(report_i)

with open('output/evaluation/classification_report.txt', 'w') as f:
    f.write('=== WORKOUT TYPE ===\n' + report_w)
    f.write('\n=== INTENSITY BAND ===\n' + report_i)

In [ ]:
# ── 3. Confusion Matrices ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_w = confusion_matrix(yw_test, yw_pred)
sns.heatmap(cm_w, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=workout_names, yticklabels=workout_names)
axes[0].set_title(f'Workout Type — F1-macro: {f1_w:.4f}', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

cm_i = confusion_matrix(yi_test, yi_pred)
sns.heatmap(cm_i, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=intensity_names, yticklabels=intensity_names)
axes[1].set_title(f'Intensity Band — F1-macro: {f1_i:.4f}', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('output/evaluation/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4. ROC Curves (One-vs-Rest) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Workout Type ROC
yw_test_bin = label_binarize(yw_test, classes=workout_classes)
for i, cls_name in enumerate(workout_names):
    fpr, tpr, _ = roc_curve(yw_test_bin[:, i], yw_proba[:, i])
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f'{cls_name} (AUC={roc_auc:.3f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('ROC — Workout Type (One-vs-Rest)', fontweight='bold')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right', fontsize=9)

# Intensity ROC
yi_test_bin = label_binarize(yi_test, classes=intensity_classes)
for i, cls_name in enumerate(intensity_names):
    fpr, tpr, _ = roc_curve(yi_test_bin[:, i], yi_proba[:, i])
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, label=f'{cls_name} (AUC={roc_auc:.3f})', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_title('ROC — Intensity Band (One-vs-Rest)', fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('output/evaluation/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5. RF vs XGBoost Comparison ──
with open('output/models/training_metrics.json') as f:
    train_metrics = json.load(f)

models_compare = ['RF (baseline)', 'XGBoost (tuned)']
accs = [train_metrics['baseline_rf']['accuracy'], acc_w]
f1s  = [train_metrics['baseline_rf']['f1_macro'], f1_w]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(2)
axes[0].bar(x, accs, color=['gray', 'steelblue'])
axes[0].set_xticks(x); axes[0].set_xticklabels(models_compare)
axes[0].set_title('Accuracy: RF vs XGBoost', fontweight='bold')
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

axes[1].bar(x, f1s, color=['gray', 'orange'])
axes[1].set_xticks(x); axes[1].set_xticklabels(models_compare)
axes[1].set_title('F1-macro: RF vs XGBoost', fontweight='bold')
axes[1].axhline(0.85, color='red', linestyle='--', label='Target 0.85')
axes[1].legend()
for i, v in enumerate(f1s):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('output/evaluation/comparison_rf_vs_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

improvement_pp = (f1_w - train_metrics['baseline_rf']['f1_macro']) * 100
print(f'XGBoost improvement vs RF: {improvement_pp:+.2f} pp F1-macro')

In [ ]:
# ── 6. Save Final Metrics ──
final_metrics = {
    'workout_type': {
        'accuracy': round(float(acc_w), 4),
        'f1_macro': round(float(f1_w), 4),
        'f1_weighted': round(float(f1_w_weighted), 4),
        'target_pass': bool(f1_w >= 0.85),
    },
    'intensity': {
        'accuracy': round(float(acc_i), 4),
        'f1_macro': round(float(f1_i), 4),
        'f1_weighted': round(float(f1_i_weighted), 4),
        'target_pass': bool(f1_i >= 0.80),
    },
    'vs_baseline_rf': {
        'rf_f1': train_metrics['baseline_rf']['f1_macro'],
        'xgb_f1': round(float(f1_w), 4),
        'improvement_pp': round(float(improvement_pp), 2),
    },
    'n_test': len(X_test),
}
with open('output/evaluation/metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

print('✅ Evaluation selesai.')
print(json.dumps(final_metrics, indent=2))